### 03 - Parallel Data processing

In this notebook, we will process parallel data (Ojibwe - English) sentences and select sentences that has ambiguity.

In [1]:
import pandas as pd
from fst_runtime.fst import Fst

In [2]:
# initialize FST parser
OJIBWE_BINARY_FILE = "../data/fst/ojibwe.att"
fst = Fst(OJIBWE_BINARY_FILE)

In [3]:
def fst_parse_word(input_word:str, fst_parser:Fst) -> list[str]:
    """ Parse an Ojibwe word and return a list of analyses"""
    fst_analyses = fst_parser.up_analysis(wordform=input_word)
    return [item.output_string
            for item in fst_analyses
            ] 

In [4]:
# test Fst parser
word = "niwanitoosiinan"
fst_parse_word(input_word=word, fst_parser=fst)

['wanitoon+VTI+Ind+Neg+Neu+1SgSubj+0PlObj',
 'wanitoon+VTI+Ind+Neg+Neu+1SgSubj+0SgObj']

In [5]:
PUNCTUTATIONS = ".,!()?$"

def tokenize(ojibwe_sentence:str) -> list[str]:
    "Tokenize Ojibwe sentence, separate symbols like . , !"
    
    raw_tokens = ojibwe_sentence.lower().split()

    output = []
    for i, item in enumerate(raw_tokens):
        if item[0] in PUNCTUTATIONS:
            # word starts with a character in PUNCTUATIONS
            output.extend([item[0], item[1:]])
        elif item[-1] in PUNCTUTATIONS:
            # word ends with a character in PUNCTUATIONS
            output.extend([item[:-1], item[-1]])
        else:
           output.append(item) 
    
    return output 

# ojibwe_sentence = "Mii iwidi gii-waabamag waawaashkeshi jiigaakwaa." # "That's there I saw the deer, near the woods."
ojibwe_sentence = "Odaanaang bimibatoowan odayan gaa-bimaagonebizod."
tokens = tokenize(ojibwe_sentence=ojibwe_sentence)
tokens


['odaanaang', 'bimibatoowan', 'odayan', 'gaa-bimaagonebizod', '.']

In [6]:
def fst_parse_sentence(input_words:list[str], fst_parser:Fst) -> list:
    """ Parse an Ojibwe sentence (list of words) and return a nested list of analyses"""
    return [{"word_form": word,
             "fst_analyses": fst_parse_word(word, fst_parser=fst_parser)
            }
            for word in input_words
            ]

In [7]:

sentence_fst_outputs = fst_parse_sentence(input_words=tokens, fst_parser=fst)
sentence_fst_outputs

[{'word_form': 'odaanaang', 'fst_analyses': ['odaanaang+ADVLoc']},
 {'word_form': 'bimibatoowan',
  'fst_analyses': ['bimibatoo+VAI+Ind+Pos+Neu+3PlObvSubj',
   'bimibatoo+VAI+Ind+Pos+Neu+3SgObvSubj']},
 {'word_form': 'odayan',
  'fst_analyses': ['day+NAD+ObvPl+3SgProxPoss',
   'day+NAD+ObvSg+3SgProxPoss',
   'ayaan+VTI+Ind+Pos+Neu+3SgProxSubj+0SgObj']},
 {'word_form': 'gaa-bimaagonebizod',
  'fst_analyses': ['PVSub/gaa+bimaagonebizo+VAI+Cnj+Pos+Neu+3SgProxSubj']},
 {'word_form': '.', 'fst_analyses': []}]

In [8]:
def sentence_has_ambiguity(sentence:str) -> tuple:
    """User FST parser to parse sentence and returns if the sentence has ambiguity in any word"""
    tokens = tokenize(ojibwe_sentence=sentence)
    sentence_fst_outputs = fst_parse_sentence(input_words=tokens, fst_parser=fst)
    for item in sentence_fst_outputs:
        if len(item.get("fst_analyses", [])) > 1:
            return (True, sentence_fst_outputs, item)
    
    return (False, sentence_fst_outputs, None)

assert sentence_has_ambiguity("niwanitoosiinan")[0] == True
assert sentence_has_ambiguity("niwaabamaa")[0] == False
assert sentence_has_ambiguity("Gabe-giizhig nimawadisaanaan Anangokwe.")[0] == True 
assert sentence_has_ambiguity("Niizho-biboonagad gii-waabamag ishkwaaj nishiime.")[0] == True 
assert sentence_has_ambiguity("Niminwendaan waabaminaan.")[0] == False 
 
print("Passed")

    

Passed


In [9]:
sentence = "Mii imaa bemi-izhising oodena jiigi-zaaga'igan." # The town is by the lake.
sentence_has_ambiguity(sentence=sentence)

(True,
 [{'word_form': 'mii', 'fst_analyses': ['mii+ADVPred']},
  {'word_form': 'imaa', 'fst_analyses': ['imaa+ADVLoc']},
  {'word_form': 'bemi-izhising',
   'fst_analyses': ['PVDir/bimi+izhisin+VII+Pcp+Pos+Neu+0SgSubj+0SgHead']},
  {'word_form': 'oodena', 'fst_analyses': []},
  {'word_form': "jiigi-zaaga'igan",
   'fst_analyses': ["PNLex/jiigi+zaaga'igan+NI+Sg",
    "jiigi-zaaga'igan+ADVLoc"]},
  {'word_form': '.', 'fst_analyses': []}],
 {'word_form': "jiigi-zaaga'igan",
  'fst_analyses': ["PNLex/jiigi+zaaga'igan+NI+Sg", "jiigi-zaaga'igan+ADVLoc"]})

In [10]:
FILENAME = "../data/parallel_data/example_sentences.csv"
dataset = pd.read_csv(FILENAME, on_bad_lines="skip")
print("Row counts =", len(dataset))
dataset.head()

Row counts = 4874


,Ojibwe,English,Speaker,Audio Link
0,Odaanaang bimibatoowan odayan gaa-bimaagonebizod.,The snowmobiler's dog is running behind him.,nj,https://s3.amazonaws.com/ojibwe-audio-transcod...
1,Mishawagaam waasaashkaa.,There are whitecaps out in the lake.,nj,https://s3.amazonaws.com/ojibwe-audio-transcod...
2,Gichi-onzaamaanimad. Waasaashkaa iwe zaaga'igan.,We have a heavy wind. The lake is full of whit...,es,https://s3.amazonaws.com/ojibwe-audio-transcod...
3,Gii-nameshin a'aw ginebig o'omaa gii-pimi-ayaa...,The trail of the snake shows it must have pass...,es,https://s3.amazonaws.com/ojibwe-audio-transcod...
4,Oshkiinamoog gaa-gii-pimi-miikanaakewaad. Gana...,There are fresh tracks of people making a (sno...,nj,https://s3.amazonaws.com/ojibwe-audio-transcod...


In [11]:
ojibwe_list = []
english_list = []
ambiguity_word_list = []
fst_readings_list = []

max_lines = len(dataset) # use it to process entire dataset
# max_lines = 100 


for i in range(max_lines):
    print(f"[Ambiguity count = {len(ojibwe_list)}]. Processing sentence #{i+1} / {max_lines} = {(i+1)*100/max_lines:.0f}% ...", end="\r")
    ojibwe_sentence = dataset.iloc[i]["Ojibwe"]
    # if sentence_has_ambiguity(sentence=ojibwe_sentence):
    has_ambiguity, fst_readings, ambiguity_word = sentence_has_ambiguity(sentence=ojibwe_sentence)
    if has_ambiguity:
        english_sentence = dataset.iloc[i]["English"]
        ojibwe_list.append(ojibwe_sentence)
        english_list.append(english_sentence)
        ambiguity_word_list.append(ambiguity_word)
        fst_readings_list.append(fst_readings)
        
    
print()
print("Number of sentence with ambiguity =", len(ojibwe_list))

[Ambiguity count = 2640]. Processing sentence #4874 / 4874 = 100% ...
Number of sentence with ambiguity = 2641


In [12]:
# create a pandas dataframe and write to output
output_dataset = pd.DataFrame(
    {"ojibwe": ojibwe_list,
     "english": english_list, 
     "word_with_ambiguity": ambiguity_word_list,
     "fst_readings": fst_readings_list
     }
)
print("Count =", len(output_dataset))
output_dataset.head()

Count = 2641


,ojibwe,english,word_with_ambiguity,fst_readings
0,Odaanaang bimibatoowan odayan gaa-bimaagonebizod.,The snowmobiler's dog is running behind him.,"{'word_form': 'bimibatoowan', 'fst_analyses': ...","[{'word_form': 'odaanaang', 'fst_analyses': ['..."
1,Gii-nameshin a'aw ginebig o'omaa gii-pimi-ayaa...,The trail of the snake shows it must have pass...,"{'word_form': 'gii-pimi-ayaagwen', 'fst_analys...","[{'word_form': 'gii-nameshin', 'fst_analyses':..."
2,Oshkiinamoog gaa-gii-pimi-miikanaakewaad. Gana...,There are fresh tracks of people making a (sno...,"{'word_form': 'gii-pimi-ayaawag', 'fst_analyse...","[{'word_form': 'oshkiinamoog', 'fst_analyses':..."
3,Wenda-gabe-ishkwaa-naawakwe babaam... obabaama...,The hunter has been tracking that deer all aft...,"{'word_form': 'wenda-gabe-ishkwaa-naawakwe', '...","[{'word_form': 'wenda-gabe-ishkwaa-naawakwe', ..."
4,Owii-ikoshimaan iniw mitigoon gaa-aazhawishini...,He's going to remove the log that was lying ac...,"{'word_form': 'owii-ikoshimaan', 'fst_analyses...","[{'word_form': 'owii-ikoshimaan', 'fst_analyse..."


In [13]:
print(output_dataset.iloc[0])

ojibwe                 Odaanaang bimibatoowan odayan gaa-bimaagonebizod.
english                     The snowmobiler's dog is running behind him.
word_with_ambiguity    {'word_form': 'bimibatoowan', 'fst_analyses': ...
fst_readings           [{'word_form': 'odaanaang', 'fst_analyses': ['...
Name: 0, dtype: object


In [14]:
# write to output csv file
OUTPUT_FILENAME = "../data/parallel_data/sentences_with_ambiguity.csv"
print(f"Writing to file {OUTPUT_FILENAME}")
output_dataset.to_csv(OUTPUT_FILENAME, index=False)
print("Completed.")


Writing to file ../data/parallel_data/sentences_with_ambiguity.csv
Completed.
